In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [2]:
### creating data points
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['20ad44ef-cf8f-46c9-ab75-177757e706a3',
  'a8515736-c91a-4217-8710-4e5cfde92945',
  '724d6b8a-0a1f-4059-8250-38a5158705a4',
  '614394c6-9435-434a-a7a4-df7d46bb2a4c',
  'afb2b0e2-5c28-4e10-ae34-4e849535ca6c'],
 'count': 5,
 'as_of': '2026-06-18T17:04:42.502103529Z'}

In [5]:
import os
import openai
from dotenv import load_dotenv
from langsmith import wrappers

load_dotenv()

openai_client = wrappers.wrap_openai(openai.OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
))

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
{inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT:
Grade:
"""
    response = openai_client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # upgraded to the more capable model for better grading
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ]
    ).choices[0].message.content

    return response.strip() == "CORRECT"

In [6]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

Run Evaluations

In [7]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "llama-3.3-70b-versatile", instructions: str = default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [8]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [10]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-groqllama-chatbot"
)

View the evaluation results for experiment: 'openai-groqllama-chatbot-7e194fd7' at:
https://smith.langchain.com/o/3862e55c-6ee7-4958-a0e8-3cafc5300cf1/datasets/f6c520ec-da3b-4e82-be33-4317f1c71dbe/compare?selectedSessions=431eb54b-1b58-4132-8f71-6f726cec8c4a




5it [00:02,  1.84it/s]


In [13]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="llama-3.1-8b-instant")}

In [14]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-groq-gemma-chatbot"
)

View the evaluation results for experiment: 'openai-groq-gemma-chatbot-2311e5f9' at:
https://smith.langchain.com/o/3862e55c-6ee7-4958-a0e8-3cafc5300cf1/datasets/f6c520ec-da3b-4e82-be33-4317f1c71dbe/compare?selectedSessions=3188fd9f-02bb-4f7a-8a62-d91bc24ee9e5




5it [00:03,  1.37it/s]
